# REINFORCE on CartPole-v1

REINFORCE is a method used in reinforcement learning to improve how decisions are made. It learns by trying actions and then adjusting the chances of those actions based on the total reward received afterward.

Unlike other methods that estimate how good each action is REINFORCE directly learns the best way to choose actions. This makes it especially useful for tasks where there are many possible actions or continuous choices and when it is hard to estimate the value of each action.

In this example we will train a policy network to solve a basic environment such as CartPole from OpenAI's gym. The aim is to use REINFORCE to directly optimize the policy without using value function approximations.

## Step 1: Setup the environment

In [1]:
import gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

env = gym.make('CartPole-v1', new_step_api=True)
obs_space = env.observation_space.shape[0]
act_space = env.action_space.n

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist

### Set the hyperparameters

In this step we define hyperparameters for the algorithm like discount factor gamma, the learning rate, number of episodes and batch size. These hyperparameters control how the algorithm behaves during training.

In [2]:
gamma = 0.99
learning_rate = 0.01
num_episodes = 1000
batch_size = 64

## Step 2: Define the Policy Network

We define the policy network as a simple neural network with two dense layers. The input to the network is the state and the output is a probability distribution over the actions (softmax output). The network learns the policy that maps states to action probabilities.

In [3]:
class PolicyNetwork(tf.keras.Model):
    def __init__(self, hidden_units=128):
        super(PolicyNetwork, self).__init__()
        self.dense1 = layers.Dense(hidden_units, activation='relu')
        self.dense2 = layers.Dense(env.action_space.n, activation='softmax')

    def call(self, state):
        x = self.dense1(state)
        return self.dense2(x)

## Step 4: Initialize the Policy and Optimizer

Here, we initialize the policy network and the Adam optimizer. The optimizer is used to update the weights of the policy network during training.

In [4]:
policy = PolicyNetwork()
optimizer = tf.keras.optimizers.Adam(learning_rate)

## Step 5: Compute Returns

In reinforcement learning, the return $G_t$ is the discounted sum of future rewards. This function computes the return for each time step $t$, based on the rewards collected during the episode.

In [5]:
def compute_returns(rewards, gamma):
    returns = np.zeros_like(rewards, dtype=np.float32)
    running_return = 0
    for t in reversed(range(len(rewards))):
        running_return = rewards[t] + gamma * running_return
        returns[t] = running_return
    return returns

## Step 6: Define Training Step

The training step computes the gradients of the policy network using the log of action probabilities and the computed returns. The loss is the negative log-likelihood of the actions taken, weighted by the return. The optimizer updates the policy network’s parameters to maximize the expected return.

In [6]:
def train_step(states, actions, returns):
    with tf.GradientTape() as tape:
        # Calculate the probability of each action taken
        action_probs = policy(states)
        action_indices = np.array(actions, dtype=np.int32)

        # Gather the probabilities for the actions taken
        action_log_probs = tf.math.log(tf.reduce_sum(action_probs * tf.one_hot(action_indices, env.action_space.n), axis=1))

        # Calculate the loss (negative log likelihood * returns)
        loss = -tf.reduce_mean(action_log_probs * returns)

    grads = tape.gradient(loss, policy.trainable_variables)
    optimizer.apply_gradients(zip(grads, policy.trainable_variables))

## Step 7: Training Loop

The training loop collects experiences from episodes and then performs training in batches. The policy is updated after each batch of experiences. In each episode, we record the states, actions and rewards and then compute the returns. The policy is updated based on these returns.

In [12]:
for episode in range(num_episodes):
    state, _ = env.reset(return_info=True)
    done = False
    states, actions, rewards = [], [], []

    while not done:
        state_input = np.array(state, dtype=np.float32).reshape(1, -1)
        probs = policy(state_input).numpy()[0]
        action = np.random.choice(act_space, p=probs)

        next_state, reward, done, _ = env.step(action)

        states.append(state_input[0])
        actions.append(action)
        rewards.append(reward)
        state = next_state

    # After episode ends
    returns = compute_returns(rewards, gamma)
    returns = (returns - np.mean(returns)) / (np.std(returns) + 1e-9)

    states_batch = np.vstack(states)
    train_step(states_batch, actions, returns)

    if episode % 100 == 0:
        print(f"Episode {episode}/{num_episodes}")

Episode 0/1000
Episode 100/1000
Episode 200/1000
Episode 300/1000
Episode 400/1000
Episode 500/1000
Episode 600/1000
Episode 700/1000
Episode 800/1000
Episode 900/1000


## Step 8: Testing the Trained Agent
After training the agent, we evaluate its performance by letting it run in the environment without updating the policy. The agent chooses actions based on the highest probabilities.

In [15]:
state, _ = env.reset(return_info=True)
done = False
total_reward = 0

while not done:
    state_input = np.array(state, dtype=np.float32).reshape(1, -1)
    probs = policy(state_input).numpy()[0]
    action = np.argmax(probs)

    next_state, reward, done, _ = env.step(action)

    total_reward += reward
    state = next_state

print(f"Test Total Reward: {total_reward}")

Test Total Reward: 500.0
